# Фильтруем данные — отбираем из больших датасетов только корейских исполнителей по списку, который мы получили ранее

In [2]:
import pandas as pd
from datasets import load_dataset

In [16]:
# Загружаем большой датасет с HF — в нем очень много данных из спотифая (в том числе музыкальные признаки)
dataset = load_dataset('GildasLeDrogoff/spotify-huge-track-analysis-dataset', split='train')

df = dataset.to_pandas()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56277664 entries, 0 to 56277663
Data columns (total 27 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   track_id                    str    
 1   artist_name                 str    
 2   track_name                  str    
 3   album_name                  str    
 4   album_release_date          object 
 5   duration_ms                 uint32 
 6   explicit                    uint8  
 7   track_number                uint16 
 8   disc_number                 uint16 
 9   track_popularity            uint8  
 10  album_popularity            uint8  
 11  track_vs_album_popularity   float64
 12  artist_popularity           uint8  
 13  artist_followers            uint64 
 14  album_vs_artist_popularity  float64
 15  tempo                       float64
 16  key                         uint8  
 17  mode                        uint8  
 18  danceability                float64
 19  energy                      fl

In [17]:
# пробуем фильтровать по одному исполнителю
skz = df[df['artist_name'] == 'Stray Kids']
skz.sort_values(by='track_popularity')

,track_id,artist_name,track_name,album_name,album_release_date,duration_ms,explicit,track_number,disc_number,track_popularity,...,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,energy_danceability_score
41743031,2L0io9ZpF32WaPSNj8jacz,Stray Kids,TOP -Japanese ver.-,ALL IN,2020-10-27,188826,0,6,1,1,...,1,0.611,0.967,-1.576,0.1470,0.001390,0.000,0.3020,0.414,0.590837
50718583,2uTsoFfSXVb1aFT5TKdvR9,Stray Kids,SLUMP -Japanese ver.-,ALL IN,2020-10-27,136133,0,7,1,1,...,1,0.598,0.813,-3.411,0.2460,0.158000,0.000,0.1250,0.686,0.486174
44583088,1rrx7dRCOfv5I0KvNVFNjC,Stray Kids,SLUMP (Instrumental),TOP -Japanese ver.-,2020-05-26,134840,0,4,1,1,...,1,0.576,0.666,-7.025,0.0303,0.001480,0.119,0.0943,0.439,0.383616
42669420,3pHfgPdWkq2BSl12AdCgAi,Stray Kids,BANG CHAN'S VOICE MEMO,Stray Kids WORLD TOUR [dominATE SEOUL],2024-09-16,16181,0,4,1,1,...,1,0.752,0.130,-15.879,0.8930,0.728000,0.000,0.1620,0.964,0.097760
24774777,3B3ksm2lV9KnwSSHmEXuh1,Stray Kids,TOP (Instrumental),TOP -Japanese ver.-,2020-05-26,185826,0,3,1,1,...,1,0.588,0.927,-3.066,0.0383,0.000042,0.940,0.1630,0.205,0.545076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21330784,63irPUP3xB74fHdw1Aw9zR,Stray Kids,MANIAC,ODDINARY,2022-03-18,182757,0,2,1,75,...,1,0.581,0.800,-2.613,0.4110,0.154000,0.000,0.0697,0.709,0.464800
18772955,37ozVDmL5b6NNVWFYgAlkz,Stray Kids,Come Play (from the series Arcane League of Le...,Come Play (from the series Arcane League of Le...,2024-10-16,161666,0,1,1,77,...,1,0.565,0.867,-5.465,0.0964,0.003040,0.000,0.2570,0.424,0.489855
47095673,1OG1NoKpZZLrMqMYCk9m84,Stray Kids,LALALALA,ROCK-STAR,2023-11-10,182224,0,2,1,77,...,1,0.705,0.849,-2.595,0.0493,0.064700,0.000,0.6530,0.609,0.598545
30799497,5emQyqYHyUOcuS3nsuC0sm,Stray Kids,Walkin On Water,HOP,2024-12-13,148810,0,1,1,79,...,1,0.557,0.958,-1.593,0.2670,0.051900,0.000,0.1630,0.910,0.533606


In [18]:
# подгружаем список исполнителей
sales_df = pd.read_csv('kpop_sales.csv', header=0, index_col=0)
sales_df.head()

,title,group type,availability,sales,first places
0,Seventeen,male group,All data available,46049635,19
1,BTS,male group,All data available,45640806,23
2,Stray Kids,male group,All data available,34341864,11
3,TXT,male group,All data available,21831591,5
4,NCT DREAM,male group,All data available,21820186,5


In [19]:
# подготавливаем список исполнителей
artist_lst = sales_df['title'].to_list()
artist_set = {a for a in artist_lst} # не приводим к нижнему регистру, регистр важен!!

df['artist_name'] = df['artist_name'].astype('category')

# фильтрация
mask = df['artist_name'].isin(artist_set)
kpop_df = df[mask].copy()

# подготавливаем sales_df для слияния
sales_merge = sales_df.copy()
sales_merge['artist_lower'] = sales_merge['title'].str.lower()
# переименовываем 'title' в 'artist_name', чтобы при merge не было дублирования
sales_merge = sales_merge.rename(columns={'title': 'artist_name_from_sales'})

# создаем в отфильтрованном df колонку для ключа слияния
kpop_df['artist_lower'] = kpop_df['artist_name'].str.lower()

# мерджим
result = kpop_df.merge(
    sales_merge[['artist_lower', 'group type', 'availability', 'sales', 'first places']],
    on='artist_lower',
    how='left'
)

# удаляем временные колонки
result.drop(columns=['artist_lower'], inplace=True)

print(len(result))

114077


In [20]:
# несовпадения по исполнителям
res_names = result['artist_name'].to_list()
for i, name in enumerate(artist_lst):
    if name not in res_names:
        print(f'{i}) {name}')

56) kangdaniel
84) ALPHA DRIVE ONE
121) JEONGHAN X WONWOO
167) J.Y.Park
175) Wheesung
179) AND2BLE
193) NCT JNJM
198) Park Jihyun
209) Red Velvet - IRENE&SEULGI
233) NOWZ
260) MODYSSEY
291) SHOWNU X HYUNGWON
304) XngHan&Xoul
315) FLARE U
328) LNGSHOT
330) KANG SEUNG YOON
335) BB GIRLS
356) BEOMGYU
370) CHUEI LI YU
383) Kim Dong Wan of NELL
384) Jay Park + LNGSHOT
390) KANGMIN
394) LEE CHAEYEON
405) RYEWOOK
414) KEYVITUP
422) TUNEXX
457) Le'v
471) JANG HANEUM
497) ChaDongHyeop (DRIPPIN)
522) NINE.I
533) LEEHI
550) MADEIN S
554) Bang Yongguk
557) Jung Ilhoon
559) Seohyun
560) hrtz.wav
561) LONGGUO & SHIHYUN
562) DAILY:DIRECTION
570) Jin Longguo
571) Choi Sooho
573) ZOONIZINI
574) Ahn Sunghoon
584) Forte Di Quattro
591) Secret Number
592) Junjin
593) USPEER
594) G.na
595) M.C The Max
596) Minho
601) Gil Byeongmin
602) Hellovenus
604) Giriboy
605) Pixy
606) Dongwoo
612) Yim Jaebum
619) Code Kunst
621) B.O.Y
622) Heejin
625) Baby DONT Cry
627) YUHZ
630) Kim Gunmo
632) 4Men
635) Lee Seunghoo

In [21]:
# фильтруем по дате (с 2011 года)
target_date = pd.to_datetime('2011-01-01')
df_2011 = result[pd.to_datetime(result['album_release_date']) >= target_date]
df_2011.to_csv('all_kpop_2011-2025.csv')
df_2011.info()

<class 'pandas.DataFrame'>
Index: 94601 entries, 0 to 114076
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   track_id                    94601 non-null  str     
 1   artist_name                 94601 non-null  category
 2   track_name                  94601 non-null  str     
 3   album_name                  94601 non-null  str     
 4   album_release_date          94601 non-null  object  
 5   duration_ms                 94601 non-null  uint32  
 6   explicit                    94601 non-null  uint8   
 7   track_number                94601 non-null  uint16  
 8   disc_number                 94601 non-null  uint16  
 9   track_popularity            94601 non-null  uint8   
 10  album_popularity            94601 non-null  uint8   
 11  track_vs_album_popularity   94601 non-null  float64 
 12  artist_popularity           94601 non-null  uint8   
 13  artist_followers            946

In [22]:
df_2011.describe()

,duration_ms,explicit,track_number,disc_number,track_popularity,album_popularity,track_vs_album_popularity,artist_popularity,artist_followers,album_vs_artist_popularity,...,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,energy_danceability_score,sales,first places
count,9.460100e+04,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,9.460100e+04,94601.000000,...,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,94601.000000,9.427300e+04,94273.000000
mean,2.136543e+05,0.074164,5.073636,1.035666,17.159438,20.859177,1.029521,44.831968,2.370963e+06,0.996635,...,0.680720,-6.474453,0.096484,0.275502,0.097298,0.212987,0.494154,0.425383,2.022225e+06,0.746067
std,8.008489e+04,0.262039,5.836831,0.307649,15.146712,15.753867,0.609047,20.034338,8.374071e+06,0.413474,...,0.217862,4.610064,0.104932,0.289842,0.262203,0.185231,0.241199,0.178023,6.200938e+06,2.941249
min,1.611400e+04,0.000000,1.000000,1.000000,1.000000,1.000000,0.025641,0.000000,0.000000e+00,0.017366,...,0.000131,-49.847000,0.000000,0.000000,0.000000,0.007430,0.000000,0.000000,2.870000e+02,0.000000
25%,1.845450e+05,0.000000,1.000000,1.000000,5.000000,8.000000,0.745098,33.000000,2.767700e+04,0.869010,...,0.545000,-7.511000,0.036900,0.035000,0.000000,0.098800,0.301000,0.295528,1.330900e+04,0.000000
50%,2.093870e+05,0.000000,3.000000,1.000000,13.000000,18.000000,1.000000,47.000000,2.251790e+05,1.000000,...,0.728000,-5.331000,0.054600,0.155000,0.000000,0.135000,0.498000,0.452600,1.284120e+05,0.000000
75%,2.376850e+05,0.000000,7.000000,1.000000,26.000000,31.000000,1.144330,58.000000,1.188102e+06,1.111538,...,0.856000,-3.920000,0.107000,0.461000,0.000116,0.281000,0.686000,0.564608,1.032476e+06,0.000000
max,5.531527e+06,1.000000,158.000000,10.000000,93.000000,87.000000,19.895604,89.000000,7.842046e+07,9.167582,...,1.000000,2.272000,0.964000,0.996000,1.000000,1.000000,0.996000,0.912518,4.604964e+07,23.000000


In [23]:
# приводим track_name к нижнему регистру, удаляем дубликаты
df_2011['track_name_lower'] = df_2011['track_name'].str.lower()
df_2011.drop_duplicates(subset=['artist_name', 'track_name_lower'], keep='first', inplace=True)

# удаляем временную колонку
df_2011.drop(columns=['track_name_lower'], inplace=True)

print(f"После удаления дубликатов: {len(df_2011)} строк")

После удаления дубликатов: 72677 строк


In [42]:
# балансировка по числу треков
artist_counts = df_2011['artist_name'].value_counts()
max_tracks = 50

parts = []

for _, group in df_2011.groupby('artist_name'):
    parts.append(
        group.sample(
            n=min(len(group), max_tracks),
            random_state=42
        )
    )

df_2011_balanced = pd.concat(parts, ignore_index=True)

print(f"Строк после балансировки: {len(df_2011_balanced)}")

Строк после балансировки: 32297


In [43]:
df_2011_balanced.describe()

,duration_ms,explicit,track_number,disc_number,track_popularity,album_popularity,track_vs_album_popularity,artist_popularity,artist_followers,album_vs_artist_popularity,...,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,energy_danceability_score,sales,first places
count,3.229700e+04,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,3.229700e+04,32297.000000,...,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,32297.000000,3.224700e+04,32247.000000
mean,2.138555e+05,0.069976,4.248785,1.024491,17.048952,20.075951,1.005918,38.963495,1.015129e+06,0.994093,...,0.671794,-6.223331,0.093937,0.288130,0.103456,0.202332,0.495201,0.424246,9.470054e+05,0.287500
std,8.364839e+04,0.255110,4.500750,0.254001,14.948302,15.402122,0.558919,18.631838,4.235828e+06,0.410894,...,0.215842,3.779349,0.106726,0.290533,0.265994,0.169171,0.240474,0.178127,3.571439e+06,1.658285
min,1.611400e+04,0.000000,1.000000,1.000000,1.000000,1.000000,0.037037,0.000000,0.000000e+00,0.017366,...,0.000131,-48.081000,0.000000,0.000001,0.000000,0.007430,0.000000,0.000000,2.870000e+02,0.000000
25%,1.855330e+05,0.000000,1.000000,1.000000,5.000000,7.000000,0.714286,27.000000,1.666700e+04,0.875912,...,0.528000,-7.503000,0.035800,0.043300,0.000000,0.099300,0.299000,0.288933,7.502000e+03,0.000000
50%,2.084800e+05,0.000000,3.000000,1.000000,13.000000,17.000000,1.000000,40.000000,1.234000e+05,1.000000,...,0.716000,-5.359000,0.052200,0.172000,0.000000,0.133000,0.499000,0.448120,7.794000e+04,0.000000
75%,2.369330e+05,0.000000,6.000000,1.000000,26.000000,29.000000,1.142578,52.000000,5.842370e+05,1.105263,...,0.848000,-3.841000,0.102000,0.487000,0.000195,0.263000,0.689000,0.565326,5.171960e+05,0.000000
max,5.531527e+06,1.000000,158.000000,9.000000,88.000000,87.000000,10.726027,89.000000,7.842046e+07,7.341390,...,1.000000,2.272000,0.964000,0.996000,0.999000,1.000000,0.987000,0.912518,4.604964e+07,23.000000


In [47]:
df_2011_balanced.head()

,track_id,artist_name,track_name,album_name,album_release_date,duration_ms,explicit,track_number,disc_number,track_popularity,...,speechiness,acousticness,instrumentalness,liveness,valence,energy_danceability_score,group type,availability,sales,first places
0,1lbn4BraBUHY7jFf3oe1WY,&TEAM,Running with the pack - &TEAM ver.,First Howling : NOW,2023-11-15,185640,0,18,1,41,...,0.2950,0.15800,0.000000,0.1130,0.496,0.506522,male group,All data available,1271052.0,0.0
1,6EGXuyLBotCYrtvOtDOWrM,&TEAM,Yukiakari,Yukiakari,2024-12-16,193704,0,1,1,14,...,0.0572,0.00667,0.000000,0.2120,0.481,0.382776,male group,All data available,1271052.0,0.0
2,0TvIjzfQogOEL43VKZ9nuz,&TEAM,Dropkick - Korean ver.,雪明かり (Yukiakari),2024-12-17,178765,0,22,1,41,...,0.0497,0.17200,0.000000,0.3270,0.749,0.575757,male group,All data available,1271052.0,0.0
3,2BQEQhfUtEJXtbxaOv96Xg,&TEAM,Koegawari,Koegawari,2024-07-18,198486,0,1,1,14,...,0.0582,0.01600,0.000003,0.2310,0.798,0.674016,male group,All data available,1271052.0,0.0
4,1zTLMgEeNgwAeUidaBHo4q,&TEAM,Wonderful World,Magic Hour / Wonderful World,2025-01-08,150057,0,2,1,56,...,0.0763,0.08630,0.000000,0.0592,0.962,0.754202,male group,All data available,1271052.0,0.0


In [45]:
print(df_2011_balanced['artist_name'].value_counts())

artist_name
&TEAM            50
(G)I-DLE         50
015B             50
100%             50
2AM              50
                 ..
𝓗𝓚                0
𝔑𝔢𝔰𝔱𝔬𝔯 𝔐𝔞𝔨𝔥𝔫𝔬     0
𝕮𝖚𝖗𝖎              0
𝕽𝕺𝕸𝕰              0
𝙻𝚘𝚏𝚒 𝚜𝚑𝙰𝙸𝚗𝚢       0
Name: count, Length: 3296456, dtype: int64


In [48]:
# сохраняем
df_2011_balanced.to_excel('kpop_balanced_2011-2025.xlsx')
df_2011_balanced.to_csv('kpop_balanced_2011-2025.csv')

In [51]:
df1 = pd.read_csv("kpop_balanced_2011-2025.csv")
df2 = pd.read_csv("filtered_artists_kpop.csv")

In [52]:
# Объединяем
df = pd.concat([df1, df2], ignore_index=True)

# Удаляем дубликаты
df = df.drop_duplicates()

In [53]:
df.to_csv("merged_clean.csv", index=False)
df.to_excel("merged_clean.xlsx", index=False)

print(f"Было строк: {len(df1) + len(df2)}")
print(f"После удаления дублей: {len(df)}")

Было строк: 33667
После удаления дублей: 33667
